# Stability metrics

Mode proportion and Refusal Rate over DefaultSys and AuthSys Psycometrics scores (Closed and Open) per dataset

In [ ]:
import json

import numpy as np
import pandas as pd

from scipy import stats as scipy_stats

from llm_audit import BASE_DIR


DATASET_ORDER = [
    "F",
    "LAS",
    "D",
    "A",
    "AA",
    "RWA",
    "RWA3D",
    "KSA3",
    "ACT",
    "VSA",
    "ASC",
    "APC",
    "CSM",
    "DW",
    "BDW",
]
ABLATION_ORDER = ["default", "authsys"]
ABLATION_DISPLAY = {"default": "DefaultSys", "authsys": "AuthSys"}
SCORE_ROUND = 4
HIGH_REFUSAL_THRESH = 0.5

OLMO_ORDER = {
    "Olmo 3.1 32B": "Olmo00",
    "Olmo3 7B Base": "Olmo01",
    "Olmo3 7B Instruct SFT": "Olmo02",
    "Olmo3 7B Instruct DPO": "Olmo03",
    "Olmo3 7B Instruct RLVR": "Olmo04",
}

with open(BASE_DIR / "resources" / "input" / "models" / "final_complete.json") as f:
    models = json.load(f)

models.sort(key=lambda m: (m["group"], OLMO_ORDER.get(m["name_short"], m["name_short"])))

model_labels = [m["name"] for m in models]
short_name = {m["name"]: m["name_short"] for m in models}


# Load data
raw = pd.read_csv(BASE_DIR / "eval" / "data" / "tidy" / "construct_scores_ensemble.csv")


def _filter(df, experiment_type):
    return df[
        df["dataset"].isin(DATASET_ORDER)
        & (df["experiment_type"] == experiment_type)
        & (df["language"] == "en")
        & (~df["experiment_ablation"].isin(["reverse"]))
    ]


closed_raw = _filter(raw, "closed_question")
open_raw = _filter(raw, "open_question")


def compute_refusal_rate(df):
    g = df.groupby(["model", "dataset", "experiment_ablation"])["refusal"]
    return (g.sum() / g.count()).reset_index(name="refusal_rate")


refusal_closed = compute_refusal_rate(closed_raw)
refusal_open = compute_refusal_rate(open_raw)


def zero_refusals(df):
    df = df.copy()
    df.loc[df["refusal"] == 1, "score"] = 0.0
    return df


closed = zero_refusals(closed_raw)
open_ = zero_refusals(open_raw)


def compute_stability(df):
    def item_stability(scores):
        mode = scipy_stats.mode(scores.round(SCORE_ROUND), keepdims=True)
        return int(mode.count[0]) / len(scores)

    item_stab = (
        df.groupby(["model", "dataset", "experiment_ablation", "statement_id"])["score"]
        .apply(item_stability)
        .reset_index(name="item_stability")
    )

    all_refusal = (
        df.groupby(["model", "dataset", "experiment_ablation", "statement_id"])["refusal"]
        .apply(lambda s: (s == 1).all())
        .reset_index(name="all_refusal")
    )
    item_stab = item_stab.merge(
        all_refusal,
        on=["model", "dataset", "experiment_ablation", "statement_id"],
    )
    item_stab.loc[item_stab["all_refusal"], "item_stability"] = np.nan

    return (
        item_stab.groupby(["model", "dataset", "experiment_ablation"])["item_stability"]
        .mean()
        .reset_index(name="stability")
    )


stab_closed = compute_stability(closed)
stab_open = compute_stability(open_)


def to_wide(stab_df, decimals=3):
    idx = pd.MultiIndex.from_tuples(
        [(m, a) for m in model_labels for a in ABLATION_ORDER],
        names=["model", "experiment_ablation"],
    )
    wide = (
        stab_df.pivot(index=["model", "experiment_ablation"], columns="dataset", values="stability")
        .reindex(index=idx, columns=DATASET_ORDER)
        .round(decimals)
    )
    wide.index = pd.MultiIndex.from_tuples(
        [(short_name.get(m, m), a) for m, a in wide.index],
        names=["model", "ablation"],
    )
    return wide


def refusal_to_wide(ref_df, decimals=3):
    idx = pd.MultiIndex.from_tuples(
        [(m, a) for m in model_labels for a in ABLATION_ORDER],
        names=["model", "experiment_ablation"],
    )
    wide = (
        ref_df.pivot(index=["model", "experiment_ablation"], columns="dataset", values="refusal_rate")
        .reindex(index=idx, columns=DATASET_ORDER)
        .round(decimals)
    )
    wide.index = pd.MultiIndex.from_tuples(
        [(short_name.get(m, m), a) for m, a in wide.index],
        names=["model", "ablation"],
    )
    return wide


stability_df = pd.concat(
    {"closed": to_wide(stab_closed), "open": to_wide(stab_open)},
    names=["approach", "model", "ablation"],
)
refusal_df = pd.concat(
    {"closed": refusal_to_wide(refusal_closed), "open": refusal_to_wide(refusal_open)},
    names=["approach", "model", "ablation"],
)


def _drop_approach(df):
    return df.droplevel("approach") if "approach" in df.index.names else df


def _fmt_val(v, bold=False):
    s = f"{v:.3f}"
    return f"\\textbf{{{s}}}" if bold else s


def _make_header(datasets):
    return " & ".join(datasets) + r" & Avg"


def _tabular_shell(caption, label, col_spec, header, body_rows, footnote=""):
    parts = [
        r"\begin{table*}[htbp]",
        r"\centering",
        f"\\caption{{{caption}}}",
        f"\\label{{tab:{label}}}",
        r"\resizebox{\linewidth}{!}{%",
        f"\\begin{{tabular}}{{{col_spec}}}",
        r"\toprule",
        f"\\textbf{{Model}} & \\textbf{{Ablation}} & {header} \\\\",
        r"\midrule",
        *body_rows,
        r"\bottomrule",
        r"\end{tabular}",
        r"}%",
    ]
    if footnote:
        parts.append(r"\par\smallskip\noindent\footnotesize " + footnote)
    parts.append(r"\end{table*}")
    return "\n".join(parts)


def _make_body_rows(models, get_s, get_r, datasets, is_stab):
    rows, has_marker = [], False

    for model in models:
        s = get_s(model)
        r = get_r(model) if (is_stab and get_r is not None) else None

        vals_by_abl = {}
        for abl in ABLATION_ORDER:
            if abl not in s.index:
                continue
            cells, floats = [], []
            for d in datasets:
                v = s.loc[abl, d]
                rv = r.loc[abl, d] if (r is not None and abl in r.index) else np.nan
                if pd.isna(v):
                    cells.append("--")
                    continue
                floats.append(v)
                if is_stab:
                    cell = _fmt_val(v, bold=abs(v - 1.0) < 1e-9)
                    marker = not pd.isna(rv) and rv > HIGH_REFUSAL_THRESH
                    cells.append(cell + r"$^{\dag}$" if marker else cell)
                    has_marker |= marker
                else:
                    cells.append(_fmt_val(v, bold=v >= HIGH_REFUSAL_THRESH))
            avg = np.nanmean(floats) if floats else np.nan
            vals_by_abl[abl] = (cells, floats, avg)

        # DefaultSys / AuthSys rows
        for i, abl in enumerate(ABLATION_ORDER):
            if abl not in vals_by_abl:
                continue
            cells, _, avg = vals_by_abl[abl]
            avg_str = "--" if np.isnan(avg) else f"{avg:.3f}"
            name = model if i == 0 else ""
            rows.append(f"{name} & {ABLATION_DISPLAY[abl]} & {' & '.join(cells)} & {avg_str} \\\\")

        # Diff row (AuthSys - DefaultSys)
        if all(a in vals_by_abl for a in ABLATION_ORDER):
            diff_cells = []
            for d in datasets:
                dv = s.loc["default", d]
                av = s.loc["authsys", d]
                diff_cells.append("--" if (pd.isna(dv) or pd.isna(av)) else f"{av - dv:+.3f}")
            d_avg, a_avg = vals_by_abl["default"][2], vals_by_abl["authsys"][2]
            diff_avg = "--" if (np.isnan(d_avg) or np.isnan(a_avg)) else f"{a_avg - d_avg:+.3f}"
            rows.append(r"\rowcolor{Gray}" + f" & Diff & {' & '.join(diff_cells)} & {diff_avg} \\\\")

    return rows, has_marker


def df_to_latex(stab_df, ref_df, caption, label):
    stab_df, ref_df = _drop_approach(stab_df), _drop_approach(ref_df)
    datasets = stab_df.columns.tolist()
    ref_models = ref_df.index.get_level_values("model")

    rows, has_marker = _make_body_rows(
        models=stab_df.index.get_level_values("model").unique(),
        get_s=lambda m: stab_df.xs(m, level="model"),
        get_r=lambda m: ref_df.xs(m, level="model") if m in ref_models else None,
        datasets=datasets,
        is_stab=True,
    )
    footnote = (
        (
            r"$^\dag$ Refusal rate ${>}"
            + str(int(HIGH_REFUSAL_THRESH * 100))
            + r"\%$; stability unreliable (driven by zeroed refusals)."
        )
        if has_marker
        else ""
    )

    return _tabular_shell(caption, label, "ll" + "r" * (len(datasets) + 1), _make_header(datasets), rows, footnote)


def refusal_to_latex(ref_df, caption, label):
    ref_df = _drop_approach(ref_df)
    datasets = ref_df.columns.tolist()

    rows, _ = _make_body_rows(
        models=ref_df.index.get_level_values("model").unique(),
        get_s=lambda m: ref_df.xs(m, level="model"),
        get_r=None,
        datasets=datasets,
        is_stab=False,
    )
    return _tabular_shell(caption, label, "ll" + "r" * (len(datasets) + 1), _make_header(datasets), rows)


out_dir = BASE_DIR / "eval" / "data" / "stability"
out_dir.mkdir(parents=True, exist_ok=True)

tables = {
    "stability_closed": df_to_latex(
        stability_df.loc["closed"],
        refusal_df.loc["closed"],
        caption=r"Score stability (mode proportion), \textbf{closed-question} approach.",
        label="stability_closed",
    ),
    "stability_open": df_to_latex(
        stability_df.loc["open"],
        refusal_df.loc["open"],
        caption=r"Score stability (mode proportion), \textbf{open-question} approach.",
        label="stability_open",
    ),
    "refusal_closed": refusal_to_latex(
        refusal_df.loc["closed"],
        caption=r"Refusal rate, \textbf{closed-question} approach.",
        label="refusal_closed",
    ),
    "refusal_open": refusal_to_latex(
        refusal_df.loc["open"],
        caption=r"Refusal rate, \textbf{open-question} approach.",
        label="refusal_open",
    ),
}

for filename, tex in tables.items():
    (out_dir / f"{filename}.tex").write_text(tex)
    print(f"Wrote {out_dir / filename}.tex")

Wrote /root/llm-audit/eval/data/stability/stability_closed.tex
Wrote /root/llm-audit/eval/data/stability/stability_open.tex
Wrote /root/llm-audit/eval/data/stability/refusal_closed.tex
Wrote /root/llm-audit/eval/data/stability/refusal_open.tex
